# 1D spin-1 XY cages

This notebook mirrors the QLM cage workflow, but uses the 1D spin-1 XY chain. Instead of drawing basis states, it renders basis configurations as `pandas.DataFrame` objects.

Workflow:

1. Build a small spin-1 XY chain Hamiltonian.
2. Search for interference-caged states.
3. Select one cage record.
4. Classify the cage using the zero-mechanism report.
5. Render the cage support and nontrivial interference-zero basis states as tables.

## Imports

Keep imports centralized so the rest of the notebook reads as the actual workflow.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from __future__ import annotations

import networkx as nx
import numpy as np
from IPython.display import display
from rich import print as rich_print

from helpers import (
    classify_cage_search_result,
    basis_dataframe,
    display_basis_dataframe,
    interference_zero_dataframe
)
from qlinks.basis.configs import basis_configs_from_build_result
from qlinks.basis.sectors import sector_mask_from_build_result
from qlinks.caging import (
    CageClassificationConfig,
    CageSearchConfig,
    CageSearcher,
    classify_cage_state,
)
from qlinks.models import SpinOneXYChainModel
from qlinks.visualizer import (
    HamiltonianGraphStyle,
    HamiltonianGraphVisualizer,
)

## Model and search parameters

The spin-1 XY model has local states `m_i in {-1, 0, +1}`, no constraints, and currently only supports `builder='sparse'`. Keep `length` modest because the full Hilbert space has size `3**length`.

In [ ]:
model = SpinOneXYChainModel(
    length=6,
    boundary_condition="periodic",
    j_xy=1.0,
    h_z=2.0,
    d_z=1.0,
)

## Build the Hamiltonian

In [ ]:
build_result = model.build(
    basis_solver="dfs",
    builder="sparse",
    backend="scipy",
    sort_basis=True,
    on_missing="raise",
)

hamiltonian_matrix = build_result.hamiltonian
kinetic_matrix = build_result.kinetic
potential_matrix = build_result.potential
basis = build_result.basis
basis_configs = basis_configs_from_build_result(build_result)
sector_mask = sector_mask_from_build_result(build_result)

print("n_states =", basis.n_states)
print("H shape =", hamiltonian_matrix.shape)
print("H nnz =", hamiltonian_matrix.nnz)
print("K nnz =", kinetic_matrix.nnz)
print("V nnz =", potential_matrix.nnz)
print("K is bipartite =", nx.is_bipartite(nx.from_scipy_sparse_array(kinetic_matrix, edge_attribute="weight")))

## Search for cages

In [ ]:
searcher = CageSearcher.from_model_build_result(
    build_result,
    config=CageSearchConfig(
        search_type="type1",
        tolerance=1e-10,
        validate_full_residual=True,
        degenerate_basis_strategy="ipr",
        ipr_n_restarts=128,
        ipr_candidate_count=64,
        ipr_random_seed=0,
    ),
)

search_result = searcher.run()
print("number of cages:", len(search_result))
print("signatures:", search_result.signatures)
print("counts by signature:", search_result.counts_by_signature)

In [ ]:
signature = (0, 4)
record_index = 0
record = search_result[signature, record_index]

state_vector = record.full_state
if state_vector is None:
    state_vector = search_result.full_state_matrix()[record_index]

print("selected signature:", record.signature)
print("support size:", record.support.size)
print("energy:", record.cage_state.energy)

## Classify the selected cage

In [ ]:
report = classify_cage_state(
    record.cage_state,
    kinetic_matrix=build_result.kinetic,
    basis_configs=basis_configs,
    sector_mask=sector_mask,
    hilbert_size=search_result.hilbert_size,
    config=CageClassificationConfig(
        amplitude_tolerance=1e-10,
        cancellation_tolerance=1e-9,
        action_tolerance=1e-9,
        sector_policy="infer_support_component",
        collective_cancellation_mode="all_problematic_sum",
    ),
)

rich_print(report)

## Cage support basis states

The table contains only the basis states in the cage support. Amplitudes are the local cage-state amplitudes on that support.

In [ ]:
support_df = basis_dataframe(
    basis_configs,
    indices=np.asarray(record.support, dtype=np.int64),
    amplitudes=np.asarray(record.local_state, dtype=np.complex128),
)

display(support_df)

## Nontrivial interference-zero basis states

First render all nontrivial interference zeros, then split them by mechanism.

In [ ]:
zeros_all_df = interference_zero_dataframe(
    report,
    basis_configs,
    mechanism="all",
)

display(zeros_all_df)

## Inspect the basis

The full basis is usually too large to print in full, so the table below shows only the first few rows.

In [ ]:
full_basis_df = display_basis_dataframe(
    basis_configs,
    max_rows=30,
    title="first 20 spin-1 product states",
)

## Visualize on Hamiltonian graph

In [ ]:
graph_visualizer = HamiltonianGraphVisualizer.from_sparse_matrix(
    kinetic_matrix,
    include_self_loops=False,
    style=HamiltonianGraphStyle(
        cmap="coolwarm",
        label_vertices=True,
        vertex_size=12,
    ),
)

graph_visualizer.plot(
    backend="igraph-cairo",
    color_by="state_amplitude_real",
    state_vector=record.full_state,
    layout="mds",
    title=f"Fock-space graph colored by cage-state signed amplitude",
)

## Optional: compact summary for many records

This cell classifies several cage records and returns one row per record. It is useful before selecting a specific cage to inspect in detail.

In [ ]:
df, classification_reports = classify_cage_search_result(
    search_result,
    kinetic_matrix=build_result.kinetic,
    basis_configs=basis_configs,
    sector_mask=None,
    config=CageClassificationConfig(
        amplitude_tolerance=1e-10,
        cancellation_tolerance=1e-9,
        action_tolerance=1e-9,
        sector_policy="infer_support_component",
        collective_cancellation_mode="all_problematic_nullspace",
    ),
)

df